# 02. Feature Engineering and Selection

## 1. Introduction
This notebook transforms cleaned backlink data into a modeling-ready dataset. Missing quality metrics are imputed using domain-level medians, derived features capture negotiation dynamics and temporal patterns, and categorical variables are encoded for tree-based models. The final feature set is validated through correlation analysis before splitting into train/test partitions.

## 2. Objectives

**O1. Missing value imputation**
Fill CF/TF gaps using domain-level median values, preserving signal where partial observations exist.

**O2. Feature derivation**
Engineer price_ratio (negotiation signal) and temporal features (year, month, quarter) from raw columns.

**O3. Correlation analysis**
Compute and visualize pairwise correlations to validate feature independence and identify multicollinearity.

**O4. Encoding and selection**
Label-encode categorical features (TLD, country), define the final feature set, and create reproducible train/test splits.

## 3. Sections
| # | Section | Purpose |
|---|---------|--------|
| 4 | Environment setup | Imports, logging, paths |
| 5 | Data loading | Load cleaned Parquet from notebook 01 |
| 6 | Missing value imputation | Impute CF/TF using domain-level medians |
| 7 | Feature engineering | Add price_ratio and temporal features |
| 8 | Correlation analysis | Heatmap of numeric feature correlations |
| 9 | Categorical encoding | Label-encode TLD and country |
| 10 | Feature set and train/test split | Define final features, split data |
| 11 | Save engineered datasets | Persist train/test partitions |
| 12 | Summary | Key findings and output artifacts |

In [ ]:
import sys
from pathlib import Path


def _ensure_local_repo_src_on_path() -> None:
    for candidate in (Path.cwd() / "src", Path.cwd().parent / "src"):
        package_root = candidate / "backlink_pricing_model"
        if package_root.exists():
            candidate_str = str(candidate.resolve())
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return


_ensure_local_repo_src_on_path()

In [ ]:
import logging
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from backlink_pricing_model.core.environment import get_project_root
from backlink_pricing_model.preprocessing.data_imputation import (
    impute_metrics_by_domain,
    summarize_imputation,
)
from backlink_pricing_model.preprocessing.data_loading import save_processed
from backlink_pricing_model.preprocessing.feature_engineering import (
    add_price_ratio,
    add_temporal_features,
)
from backlink_pricing_model.visualization.importance_plots import (
    plot_correlation_heatmap,
)
from backlink_pricing_model.visualization.plots_style import save_plot

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

PROJECT_ROOT = get_project_root()
IMAGE_DIR = PROJECT_ROOT / "images" / "feature_engineering"
ENGINEERED_DATA_DIR = PROJECT_ROOT / "data" / "engineered"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
ENGINEERED_DATA_DIR.mkdir(parents=True, exist_ok=True)

logger.info("Project root: %s", PROJECT_ROOT)

## 5. Data loading

Load the cleaned backlink dataset produced by notebook 01. This dataset has valid prices, normalized categoricals, and log transforms already applied.

In [ ]:
df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "backlinks_cleaned.parquet")
logger.info("Loaded %d rows, %d columns", len(df), len(df.columns))
display(df.head())

## 6. Missing value imputation

CF and TF have significant missingness (~66%). Where a domain has multiple records with known values, the domain-level median is used to fill gaps in other records for the same domain.

In [ ]:
df_before = df.copy()
df = impute_metrics_by_domain(df)

imputation_summary = summarize_imputation(df_before, df)
logger.info("Imputation summary:")
display(imputation_summary)

## 7. Feature engineering

Add derived features that capture negotiation dynamics and temporal pricing patterns:
- **price_ratio**: `final_price / initial_price` — measures negotiation discount
- **year, month, quarter**: temporal features extracted from `date_received`

In [ ]:
df = add_price_ratio(df)
df = add_temporal_features(df)

logger.info("Columns after feature engineering: %s", list(df.columns))
display(df.describe())

## 8. Correlation analysis

Compute pairwise Pearson correlations among numeric features to check for multicollinearity and validate that the target (`log_price`) has meaningful relationships with candidate predictors.

In [ ]:
numeric_cols = [
    "final_price",
    "dr",
    "cf",
    "tf",
    "domain_traffic",
    "log_price",
    "log_traffic",
]
corr_matrix = df[numeric_cols].corr()
fig = plot_correlation_heatmap(corr_matrix, title="Numeric feature correlations")
save_plot(fig, "correlation_heatmap", str(IMAGE_DIR))

In [ ]:
Image(filename=str(IMAGE_DIR / "correlation_heatmap.png"))

_Figure 1. Pearson correlation heatmap of numeric features. DR, CF, and TF show moderate intercorrelation as expected for quality metrics. log_price correlates most strongly with DR and log_traffic._

## 9. Categorical encoding

Label-encode TLD and country for tree-based models. These encodings assign integer IDs without implying ordinal relationships, which is appropriate for decision tree splits.

In [ ]:
categorical_cols = ["tld", "country"]
label_encoders: dict[str, LabelEncoder] = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[f"{col}_encoded"] = le.fit_transform(df[col].fillna("unknown"))
    label_encoders[col] = le
    n_classes = len(le.classes_)
    logger.info("%s: %d unique values", col, n_classes)

## 10. Feature set and train/test split

Define the final modeling feature set and create a reproducible 80/20 train/test split.

In [ ]:
FEATURE_COLS = [
    "dr",
    "cf",
    "tf",
    "log_traffic",
    "tld_encoded",
    "country_encoded",
    "year",
    "month",
    "quarter",
]
TARGET = "log_price"

df_model = df.dropna(subset=[*FEATURE_COLS, TARGET])
logger.info("Modeling dataset: %d rows, %d features", len(df_model), len(FEATURE_COLS))

X = df_model[FEATURE_COLS]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)
logger.info("Train: %d | Test: %d", len(X_train), len(X_test))

## 11. Save engineered datasets

Persist the full engineered dataset and the train/test splits to `data/engineered/` for downstream modeling notebooks.

In [ ]:
save_processed(df, "backlinks_engineered", subdir="engineered")
logger.info("Saved full engineered dataset (%d rows)", len(df))

train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

save_processed(train_df, "train_df", subdir="engineered")
save_processed(test_df, "test_df", subdir="engineered")
logger.info(
    "Saved train_df (%d rows) and test_df (%d rows) to data/engineered/",
    len(train_df),
    len(test_df),
)

---

## 12. Summary

This notebook transformed the cleaned backlink dataset into a modeling-ready format with imputed metrics, derived features, and encoded categoricals.

**Key findings:**
- **Imputation:** Domain-level median imputation recovered CF/TF values for records where other observations from the same domain had known values.
- **Feature correlations:** DR, CF, and TF are moderately intercorrelated but each carries independent signal. log_traffic provides additional predictive power.
- **Temporal patterns:** Year, month, and quarter features capture market dynamics and seasonal pricing variation.
- **Final feature set:** 9 features (dr, cf, tf, log_traffic, tld_encoded, country_encoded, year, month, quarter) predicting log_price.

**Output artifacts:**
- `data/engineered/backlinks_engineered.parquet` — full engineered dataset
- `data/engineered/train_df.parquet` — training split (80%)
- `data/engineered/test_df.parquet` — test split (20%)
- `images/feature_engineering/correlation_heatmap.png` — feature correlation heatmap